#####    1.1. Exploratory Data Analysis

##### 1.2. Data Filtering & Normalize



###### 1.2.1. Null Filter

#### III. Fine-tune Hirashiba-MT-tiny-zh-vi

Mô hình [Hirashiba-MT-tiny-zh-vi](https://huggingface.co/chi-vi/hirashiba-mt-tiny-zh-vi) là mô hình dịch máy nhỏ gọn (15.1M params) được pre-train sẵn cho cặp ngôn ngữ Trung - Việt.

**Thông tin mô hình:**
- Kiến trúc: Encoder-decoder transformer
- Số tham số: 15.1 triệu
- Vocabulary size: 29,067

In [ ]:
!pip install torch sacrebleu -q

##### 3.1 Import thư viện & Đường dẫn

In [9]:
import torch
import pandas as pd
import json
import sacrebleu
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed
)
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm
import random
import gc

In [19]:
DATA_DIR = Path("/kaggle/working/zh_vi_translation")
MODEL_NAME_HIRASHIBA = "chi-vi/hirashiba-mt-tiny-zh-vi"

# Thư mục riêng cho Hirashiba
HIRASHIBA_OUTPUT_DIR = DATA_DIR / "hirashiba_output"
HIRASHIBA_CHECKPOINT_DIR = HIRASHIBA_OUTPUT_DIR / "checkpoints"
HIRASHIBA_BEST_DIR = HIRASHIBA_OUTPUT_DIR / "best_model"

# Tạo thư mục
HIRASHIBA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HIRASHIBA_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HIRASHIBA_BEST_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Hirashiba output: {HIRASHIBA_OUTPUT_DIR}')

Device : cuda
Hirashiba output: /kaggle/working/zh_vi_translation/hirashiba_output


##### 3.2. Load tokenizer và model từ HuggingFace

In [7]:
print(f"\n⏳ Loading tokenizer and model: {MODEL_NAME_HIRASHIBA}")

tokenizer_hirashiba = AutoTokenizer.from_pretrained(
    MODEL_NAME_HIRASHIBA,
    trust_remote_code=True
)

model_hirashiba = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME_HIRASHIBA,
    trust_remote_code=True
).to(DEVICE)

# Kiểm tra số tham số
n_params = sum(p.numel() for p in model_hirashiba.parameters()) / 1e6
print(f"✅ Model loaded: {n_params:.1f}M params")
print(f"   Vocab size: {len(tokenizer_hirashiba)}")


⏳ Loading tokenizer and model: chi-vi/hirashiba-mt-tiny-zh-vi


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/418k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/470k [00:00<?, ?B/s]

...

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


model.safetensors:   0%|          | 0.00/59.4M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

✅ Model loaded: 15.1M params
   Vocab size: 29067


##### 3.3. Load dữ liệu processed (train/dev/test)

In [75]:
TSV_COLS = ["zh", "vi"]

def load_tsv_data(path, max_samples=None):
    df = pd.read_csv(
        path, sep="\t", header=None, names=TSV_COLS,
        quoting=3, on_bad_lines="skip", dtype=str, engine="python",
    ).dropna().reset_index(drop=True)
    if max_samples:
        df = df.iloc[:max_samples]
    return df

df_train_hirashiba = load_tsv_data("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/train.tsv")
df_dev_hirashiba = load_tsv_data("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/dev.tsv")
df_test_hirashiba = load_tsv_data("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/test.tsv")

print(f"\nTrain: {len(df_train_hirashiba):,} | Dev: {len(df_dev_hirashiba):,} | Test: {len(df_test_hirashiba):,}")


Train: 494,000 | Dev: 3,000 | Test: 3,000


In [9]:
def df_to_hf_hirashiba(df):
    return Dataset.from_dict({"zh": df["zh"].tolist(), "vi": df["vi"].tolist()})

raw_datasets_hirashiba = DatasetDict({
    "train": df_to_hf_hirashiba(df_train_hirashiba),
    "dev": df_to_hf_hirashiba(df_dev_hirashiba),
    "test": df_to_hf_hirashiba(df_test_hirashiba),
})

print(raw_datasets_hirashiba)

DatasetDict({
    train: Dataset({
        features: ['zh', 'vi'],
        num_rows: 494000
    })
    dev: Dataset({
        features: ['zh', 'vi'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['zh', 'vi'],
        num_rows: 3000
    })
})


##### 3.4. Tokenize dữ liệu

In [10]:
MAX_LENGTH_HIRASHIBA = 128

def preprocess_function_hirashiba(examples):
    model_inputs = tokenizer_hirashiba(
        examples["zh"],
        text_target=examples["vi"],
        max_length=MAX_LENGTH_HIRASHIBA,
        truncation=True,
        padding=False,
    )
    return model_inputs

print("\n⏳ Tokenizing datasets...")
tokenized_datasets_hirashiba = raw_datasets_hirashiba.map(
    preprocess_function_hirashiba,
    batched=True,
    batch_size=1024,
    remove_columns=["zh", "vi"],
    desc="Tokenizing",
)

print(tokenized_datasets_hirashiba)


⏳ Tokenizing datasets...


Tokenizing:   0%|          | 0/494000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 494000
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})


In [11]:
data_collator_hirashiba = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_hirashiba,
    model=model_hirashiba,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if DEVICE == "cuda" else None,
)
print("✅ DataCollator ready")

✅ DataCollator ready


##### 3.5. Cấu hình training

In [15]:
training_args_hirashiba = Seq2SeqTrainingArguments(
    output_dir=str(HIRASHIBA_CHECKPOINT_DIR),
    
    # Batch size
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    
    # Optimizer
    learning_rate=3e-4,
    warmup_steps=500,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch_fused" if DEVICE == "cuda" else "adamw_torch",
    
    # Regularization
    label_smoothing_factor=0.1,
    
    # Training
    num_train_epochs=10,
    fp16=DEVICE == "cuda",
    
    # DataLoader
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    
    # Eval & Save
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Generation
    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=MAX_LENGTH_HIRASHIBA,
    
    logging_steps=100,
    report_to="none",
    seed=42,
)

print(f"\n✅ Training config:")
eff_batch = training_args_hirashiba.per_device_train_batch_size * training_args_hirashiba.gradient_accumulation_steps
print(f"   Effective batch: {eff_batch}")
print(f"   Epochs: {training_args_hirashiba.num_train_epochs}")
print(f"   Learning rate: {training_args_hirashiba.learning_rate}")


✅ Training config:
   Effective batch: 64
   Epochs: 10
   Learning rate: 0.0003


In [19]:
trainer_hirashiba = Seq2SeqTrainer(
    model=model_hirashiba,
    args=training_args_hirashiba,
    train_dataset=tokenized_datasets_hirashiba["train"],
    eval_dataset=tokenized_datasets_hirashiba["dev"],
    processing_class=tokenizer_hirashiba,
    data_collator=data_collator_hirashiba,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n✅ Trainer ready!")


✅ Trainer ready!


##### 3.6. Training với checkpoint resume

In [20]:
def find_last_checkpoint_hirashiba(ckpt_dir):
    dirs = sorted(
        [d for d in Path(ckpt_dir).iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    return str(dirs[-1]) if dirs else None

last_ckpt_hirashiba = find_last_checkpoint_hirashiba(HIRASHIBA_CHECKPOINT_DIR)
print(f"Resume: {last_ckpt_hirashiba}" if last_ckpt_hirashiba else "Train from scratch")

# ============================================
print("\n🚀 Starting training...")
train_result_hirashiba = trainer_hirashiba.train(resume_from_checkpoint=last_ckpt_hirashiba)
trainer_hirashiba.log_metrics("train", train_result_hirashiba.metrics)

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
You are resuming training from a checkpoint trained with 5.8.1 of Transformers but your current version is 5.0.0. This is not recommended and could yield to errors or unwanted behaviors.
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Resume: /kaggle/working/zh_vi_translation/zh_vi_processed/hirashiba_output/checkpoints/checkpoint-12500

🚀 Starting training...


Step,Training Loss,Validation Loss
13000,9.816112,4.892126
13500,9.797135,4.885626
14000,9.817134,4.890622
14500,9.840278,4.881933
15000,9.818708,4.873827
15500,9.682604,4.876462
16000,9.712367,4.874989
16500,9.730787,4.872084
17000,9.723900,4.865752
17500,9.759791,4.869393


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


***** train metrics *****
  epoch                    =     8.1606
  total_flos               = 12198441GF
  train_loss               =     5.8177
  train_runtime            = 1:57:15.90
  train_samples_per_second =    702.113
  train_steps_per_second   =      5.486


##### 3.7. Lưu best model và đánh giá BLEU

In [21]:
trainer_hirashiba.save_model(str(HIRASHIBA_BEST_DIR))
tokenizer_hirashiba.save_pretrained(str(HIRASHIBA_BEST_DIR))
print(f"✅ Best model saved to: {HIRASHIBA_BEST_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved to: /kaggle/working/zh_vi_translation/zh_vi_processed/hirashiba_output/best_model


In [28]:
print("\n⏳ Loading best model for inference...")
best_tokenizer_hirashiba = AutoTokenizer.from_pretrained(str(HIRASHIBA_BEST_DIR), trust_remote_code=True)
best_model_hirashiba = AutoModelForSeq2SeqLM.from_pretrained(str(HIRASHIBA_BEST_DIR), trust_remote_code=True).to(DEVICE)
best_model_hirashiba.eval()
print("✅ Loaded")

# TRANSLATE FUNCTION
@torch.inference_mode()
def translate_hirashiba(text, max_length=128, num_beams=4):
    """Dịch một câu từ Trung sang Việt"""
    inputs = best_tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)

    outputs = best_model_hirashiba.generate(
        **inputs,
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True,
    )

    return best_tokenizer_hirashiba.decode(outputs[0], skip_special_tokens=True)

# TEST INFERENCE
test_samples = [
    "我爱自然语言处理",
    "今天天气很好",
    "机器翻译是自然语言处理的重要任务",
    "专家们警告朝鲜继续推进武器计划",
    "我在学习深度学习",
]

print("\n" + "="*60)
print("🔤 TEST INFERENCE - HIRASHIBA-MT-TINY-ZH-VI")
print("="*60)

for zh in test_samples:
    vi = translate_hirashiba(zh)
    print(f"ZH: {zh}")
    print(f"VI: {vi}")
    print("-" * 50)


⏳ Loading best model for inference...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

✅ Loaded

🔤 TEST INFERENCE - HIRASHIBA-MT-TINY-ZH-VI
ZH: 我爱自然语言处理
VI: Tôi yêu xử lý ngôn ngữ tự nhiên
--------------------------------------------------
ZH: 今天天气很好
VI: Hôm nay thời tiết rất tốt
--------------------------------------------------
ZH: 机器翻译是自然语言处理的重要任务
VI: Dịch máy là một nhiệm vụ quan trọng trong xử lý ngôn ngữ tự nhiên
--------------------------------------------------
ZH: 专家们警告朝鲜继续推进武器计划
VI: Các chuyên gia cảnh báo Triều Tiên tiếp tục chương trình vũ khí
--------------------------------------------------
ZH: 我在学习深度学习
VI: Tôi đang học sâu
--------------------------------------------------


In [29]:
@torch.inference_mode()
def evaluate_bleu_hirashiba(model, test_tsv_path, tokenizer, max_len=128, max_samples=None):
    model.eval()
    hypotheses = []
    references = []

    with open(test_tsv_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    if max_samples:
        lines = lines[:max_samples]

    for line in tqdm(lines, desc='Evaluating BLEU'):
        parts = line.strip().split('\t')
        if len(parts) != 2:
            continue
        zh, vi_ref = parts[0].strip(), parts[1].strip()
        if not zh or not vi_ref:
            continue

        inputs = tokenizer(zh, return_tensors="pt", truncation=True, max_length=max_len).to(DEVICE)
        outputs = model.generate(**inputs, max_length=max_len, num_beams=4, early_stopping=True)
        vi_hyp = tokenizer.decode(outputs[0], skip_special_tokens=True)

        hypotheses.append(vi_hyp)
        references.append(vi_ref)

    # Tính BLEU
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="char")
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])

    print(f'\n=== KẾT QUẢ BLEU TRÊN TẬP TEST ({len(hypotheses):,} câu) ===')
    print(f'BLEU score     : {bleu.score:.2f}')
    print(f'BLEU char      : {bleu_char.score:.2f}')
    print(f'chrF           : {chrf.score:.2f}')
    print(f'Brevity Penalty: {bleu.bp:.4f}')
    print(f'Ratio (hyp/ref): {bleu.sys_len / bleu.ref_len:.4f}')

    print('\n--- Ví dụ ngẫu nhiên ---')
    import random
    indices = random.sample(range(len(hypotheses)), min(5, len(hypotheses)))
    for i in indices:
        print(f'ZH: {lines[i].strip().split(chr(9))[0]}')
        print(f'REF: {references[i]}')
        print(f'HYP: {hypotheses[i]}')
        print('-' * 50)

    return bleu.score

print("\n⏳ Evaluating BLEU on test set...")
hirashiba_blue = evaluate_bleu_hirashiba(
    best_model,
    str("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/test.tsv"),
    best_tokenizer
)

print(f"\n✅ Final BLEU score: {hirashiba_blue:.2f}")


⏳ Evaluating BLEU on test set...


Evaluating BLEU:   0%|          | 0/3000 [00:00<?, ?it/s]


=== KẾT QUẢ BLEU TRÊN TẬP TEST (3,000 câu) ===
BLEU score     : 41.38
BLEU char      : 63.77
chrF           : 58.12
Brevity Penalty: 0.9781
Ratio (hyp/ref): 0.9784

--- Ví dụ ngẫu nhiên ---
ZH: 现在,你说你还爱我,我才明白,你靠近,或推开我,都是在保护我.
REF: Bây giờ em nói em vẫn yêu anh anh mới hiểu được, em đến gần anh hay đẩy anh ra xa đều là đang bảo vệ anh.
HYP: Bây giờ, bạn nói rằng bạn vẫn yêu tôi, tôi mới hiểu, bạn đến gần, hoặc đẩy tôi ra, tất cả đều bảo vệ tôi.
--------------------------------------------------
ZH: 我们目前的想法和选择是我们目前经验的唯一决定因素.
REF: Những suy nghĩ và lựa chọn hiện tại của chúng tôi là yếu tố quyết định duy nhất cho trải nghiệm hiện tại của chúng tôi.
HYP: Ý tưởng và lựa chọn hiện tại của chúng tôi là yếu tố quyết định duy nhất của kinh nghiệm hiện tại của chúng tôi.
--------------------------------------------------
ZH: 注意,Java中有一些我们必须遵守的规则:
REF: Lưu ý rằng trong Java có một số quy tắc mà ta phải tuân theo:
HYP: Lưu ý rằng có một số quy tắc mà chúng ta phải tuân thủ trong Java:
-----------

In [70]:
results_hirashiba = {
    "model_name": MODEL_NAME_HIRASHIBA,                    # Tên model
    "train_samples": len(df_train_hirashiba),              # Số mẫu train
    "dev_samples": len(df_dev_hirashiba),                  # Số mẫu dev
    "test_samples": len(df_test_hirashiba),                # Số mẫu test
    "bleu_score": bleu_score_hirashiba,                    # Điểm BLEU
    "num_epochs": training_args_hirashiba.num_train_epochs,   # Số epochs
    "batch_size": training_args_hirashiba.per_device_train_batch_size,  # Batch size
    "learning_rate": training_args_hirashiba.learning_rate,  # Learning rate
    "effective_batch": training_args_hirashiba.per_device_train_batch_size * training_args_hirashiba.gradient_accumulation_steps,
    "checkpoints_dir": str(HIRASHIBA_CHECKPOINT_DIR),      # Thư mục checkpoints
}

with open(HIRASHIBA_OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results_hirashiba, f, ensure_ascii=False, indent=2)

print(f"\n✅ Results saved to: {HIRASHIBA_OUTPUT_DIR / 'results.json'}")

print("\n" + "="*60)
print("✅ HIRASHIBA-MT-TINY-ZH-VI FINISHED!")
print(f"   Model saved to: {HIRASHIBA_BEST_DIR}")
print(f"   Checkpoints saved to: {HIRASHIBA_CHECKPOINT_DIR}")
print("="*60)


✅ Results saved to: /kaggle/working/zh_vi_translation/hirashiba_output/results.json

✅ HIRASHIBA-MT-TINY-ZH-VI FINISHED!
   Model saved to: /kaggle/working/zh_vi_translation/hirashiba_output/best_model
   Checkpoints saved to: /kaggle/working/zh_vi_translation/hirashiba_output/checkpoints


#### IV. Fine-tune Helsinki-NLP/opus-mt-zh-vi

[Helsinki-NLP/opus-mt-zh-vi](https://huggingface.co/Helsinki-NLP/opus-mt-zh-vi) là mô hình dịch máy từ nhóm Helsinki-NLP, được train trên dữ liệu OPUS đa ngôn ngữ.

**Thông tin mô hình:**
- Kiến trúc: MarianMT (biến thể transformer)
- Số tham số: 144.5 triệu
- Vocabulary size: 65,001

##### 4.1. Cấu hình

In [3]:
!pip install -q sentencepiece sacrebleu

import transformers, accelerate, datasets as ds_lib
print(f"transformers : {transformers.__version__}")
print(f"accelerate   : {accelerate.__version__}")
print(f"datasets     : {ds_lib.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 3.9 MB/s eta 0:00:00
transformers : 5.0.0
accelerate   : 1.12.0
datasets     : 4.8.3


In [ ]:
import torch
import pandas as pd
import json
import sacrebleu
from pathlib import Path
from transformers import (
    MarianTokenizer,
    MarianMTModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    set_seed
)
from datasets import Dataset, DatasetDict
from tqdm.auto import tqdm
import random
import gc

# ============================================
# CẤU HÌNH ĐƯỜNG DẪN
# ============================================
DATA_DIR = Path("/kaggle/working/zh_vi_translation")

# Thư mục riêng cho Helsinki (giống Hirashiba)
HELSINKI_OUTPUT_DIR = DATA_DIR / "helsinki_output"
HELSINKI_CHECKPOINT_DIR = HELSINKI_OUTPUT_DIR / "checkpoints"
HELSINKI_BEST_DIR = HELSINKI_OUTPUT_DIR / "best_model"

# Tạo thư mục
HELSINKI_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HELSINKI_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
HELSINKI_BEST_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
print(f'Helsinki output: {HELSINKI_OUTPUT_DIR}')

# ============================================
# MODEL NAME
# ============================================
MODEL_NAME_HELSINKI = "Helsinki-NLP/opus-mt-zh-vi"

Device : cuda
Helsinki output: /kaggle/working/zh_vi_translation/helsinki_output


##### 4.2. Load Marian tokenizer và model

In [ ]:
# ============================================
# LOAD TOKENIZER VÀ MODEL
# ============================================
print(f"\n⏳ Loading tokenizer and model: {MODEL_NAME_HELSINKI}")

tokenizer_helsinki = MarianTokenizer.from_pretrained(MODEL_NAME_HELSINKI)
model_helsinki = MarianMTModel.from_pretrained(MODEL_NAME_HELSINKI).to(DEVICE)

# Kiểm tra số tham số
n_params = sum(p.numel() for p in model_helsinki.parameters()) / 1e6
print(f"✅ Model loaded: {n_params:.1f}M params")
print(f"   Vocab size: {len(tokenizer_helsinki)}")


⏳ Loading tokenizer and model: Helsinki-NLP/opus-mt-zh-vi


tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/750k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/766k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

✅ Model loaded: 144.5M params
   Vocab size: 65001


##### 4.3. Chuẩn bị dataset (train/dev/test)

In [ ]:
# ============================================
# LOAD DỮ LIỆU TSV
# ============================================
TSV_COLS = ["zh", "vi"]

def load_tsv_data_helsinki(path, max_samples=None):
    df = pd.read_csv(
        path, sep="\t", header=None, names=TSV_COLS,
        quoting=3, on_bad_lines="skip", dtype=str, engine="python",
    ).dropna().reset_index(drop=True)
    if max_samples:
        df = df.iloc[:max_samples]
    return df

df_train_helsinki = load_tsv_data_helsinki("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/train.tsv")
df_dev_helsinki = load_tsv_data_helsinki("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/dev.tsv")
df_test_helsinki = load_tsv_data_helsinki("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/test.tsv")

print(f"\nTrain: {len(df_train_helsinki):,} | Dev: {len(df_dev_helsinki):,} | Test: {len(df_test_helsinki):,}")


Train: 494,000 | Dev: 3,000 | Test: 3,000


In [ ]:
# ============================================
# DATASET PREPARATION
# ============================================
def df_to_hf_helsinki(df):
    return Dataset.from_dict({"zh": df["zh"].tolist(), "vi": df["vi"].tolist()})

raw_datasets_helsinki = DatasetDict({
    "train": df_to_hf_helsinki(df_train_helsinki),
    "dev": df_to_hf_helsinki(df_dev_helsinki),
    "test": df_to_hf_helsinki(df_test_helsinki),
})

print(raw_datasets_helsinki)

DatasetDict({
    train: Dataset({
        features: ['zh', 'vi'],
        num_rows: 494000
    })
    dev: Dataset({
        features: ['zh', 'vi'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['zh', 'vi'],
        num_rows: 3000
    })
})


##### 4.4.  Tokenize dữ liệu

In [ ]:
# ============================================
# TOKENIZE FUNCTION
# ============================================
MAX_LENGTH_HELSINKI = 128

def preprocess_function_helsinki(examples):
    model_inputs = tokenizer_helsinki(
        examples["zh"],
        text_target=examples["vi"],
        max_length=MAX_LENGTH_HELSINKI,
        truncation=True,
        padding=False,
    )
    return model_inputs

print("\n⏳ Tokenizing datasets...")
tokenized_datasets_helsinki = raw_datasets_helsinki.map(
    preprocess_function_helsinki,
    batched=True,
    batch_size=1024,
    remove_columns=["zh", "vi"],
    desc="Tokenizing",
)

print(tokenized_datasets_helsinki)


⏳ Tokenizing datasets...


Tokenizing:   0%|          | 0/494000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/3000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 494000
    })
    dev: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 3000
    })
})


In [ ]:
# ============================================
# DATA COLLATOR
# ============================================
data_collator_helsinki = DataCollatorForSeq2Seq(
    tokenizer=tokenizer_helsinki,
    model=model_helsinki,
    padding=True,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if DEVICE == "cuda" else None,
)
print("✅ DataCollator ready")

✅ DataCollator ready


##### 4.5. Cấu hình training

In [ ]:
# ============================================
# TRAINING ARGUMENTS
# ============================================
training_args_helsinki = Seq2SeqTrainingArguments(
    output_dir=str(HELSINKI_CHECKPOINT_DIR),
    
    # Batch size
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,
    gradient_checkpointing=False,
    
    # Optimizer
    learning_rate=5e-5,
    warmup_steps=500,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    optim="adamw_torch_fused" if DEVICE == "cuda" else "adamw_torch",
    
    # Regularization
    label_smoothing_factor=0.1,
    
    # Training
    num_train_epochs=5,
    fp16=DEVICE == "cuda",
    
    # DataLoader
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    
    # Eval & Save
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    
    # Generation
    predict_with_generate=True,
    generation_num_beams=4,
    generation_max_length=MAX_LENGTH_HELSINKI,
    
    logging_steps=50,
    report_to="none",
    seed=42,
)

print(f"\n✅ Training config:")
eff_batch = training_args_helsinki.per_device_train_batch_size * training_args_helsinki.gradient_accumulation_steps
print(f"   Effective batch: {eff_batch}")
print(f"   Epochs: {training_args_helsinki.num_train_epochs}")
print(f"   Learning rate: {training_args_helsinki.learning_rate}")


✅ Training config:
   Effective batch: 32
   Epochs: 5
   Learning rate: 5e-05


In [ ]:
# ============================================
# TRAINER
# ============================================
trainer_helsinki = Seq2SeqTrainer(
    model=model_helsinki,
    args=training_args_helsinki,
    train_dataset=tokenized_datasets_helsinki["train"],
    eval_dataset=tokenized_datasets_helsinki["dev"],
    processing_class=tokenizer_helsinki,
    data_collator=data_collator_helsinki,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\n✅ Trainer ready!")


✅ Trainer ready!


##### 4.6. Training với resume từ checkpoint

In [10]:
# ============================================
# 10. FIND LAST CHECKPOINT (RESUME)
# ============================================
def find_last_checkpoint_helsinki(ckpt_dir):
    dirs = sorted(
        [d for d in Path(ckpt_dir).iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-")],
        key=lambda d: int(d.name.split("-")[-1]),
    )
    return str(dirs[-1]) if dirs else None

# Giải phóng bộ nhớ
gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()

last_ckpt_helsinki = find_last_checkpoint_helsinki(HELSINKI_CHECKPOINT_DIR)

if last_ckpt_helsinki:
    print(f"\n>>> Resume từ checkpoint: {last_ckpt_helsinki}")
    # Đọc số steps đã train
    ckpt_path = Path(last_ckpt_helsinki) / "trainer_state.json"
    if ckpt_path.exists():
        with open(ckpt_path, 'r') as f:
            state = json.load(f)
            print(f"    Đã train: {state.get('global_step', 0):,} steps")
else:
    print("\n>>> Train từ đầu")

# ============================================
# 11. TRAIN
# ============================================
print("\n🚀 Starting training...")

try:
    train_result_helsinki = trainer_helsinki.train(resume_from_checkpoint=last_ckpt_helsinki)
    trainer_helsinki.log_metrics("train", train_result_helsinki.metrics)
except KeyboardInterrupt:
    print("\n⚠️ Training bị dừng! Checkpoint đã được lưu.")
    print(f"   Chạy lại cell này để resume từ checkpoint gần nhất.")
    raise

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



>>> Resume từ checkpoint: /kaggle/working/zh_vi_translation/helsinki_output/checkpoints/checkpoint-1000
    Đã train: 1,000 steps

🚀 Starting training...


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


Step,Training Loss,Validation Loss
2000,12.797529,6.207088
3000,12.550759,6.064205
4000,12.221276,5.959961
5000,12.096896,5.868089
6000,11.947867,5.804425
7000,11.838212,5.747070
8000,11.451924,5.709623
9000,11.347313,5.679591
10000,11.274895,5.643764
11000,11.386904,5.614203


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_positions.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].


***** train metrics *****
  epoch                    =        5.0
  total_flos               = 25104882GF
  train_loss               =    10.7026
  train_runtime            = 7:09:09.19
  train_samples_per_second =     95.925
  train_steps_per_second   =      1.499


##### 4.7. Lưu best model và đánh giá BLEU

In [ ]:
# ============================================
# SAVE BEST MODEL
# ============================================
print("\n💾 Saving best model...")
trainer_helsinki.save_model(str(HELSINKI_BEST_DIR))
tokenizer_helsinki.save_pretrained(str(HELSINKI_BEST_DIR))
print(f"✅ Best model saved to: {HELSINKI_BEST_DIR}")


💾 Saving best model...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Best model saved to: /kaggle/working/zh_vi_translation/helsinki_output/best_model


In [ ]:
# ============================================
# LOAD BEST MODEL FOR INFERENCE
# ============================================
print("\n⏳ Loading best Helsinki model for inference...")
best_tokenizer_helsinki = MarianTokenizer.from_pretrained(str(HELSINKI_BEST_DIR))
best_model_helsinki = MarianMTModel.from_pretrained(str(HELSINKI_BEST_DIR)).to(DEVICE)
best_model_helsinki.eval()
print("✅ Loaded")

# ============================================
# TRANSLATE FUNCTION
# ============================================
@torch.inference_mode()
def translate_helsinki(text, max_length=128, num_beams=4):
    inputs = best_tokenizer_helsinki(text, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    outputs = best_model_helsinki.generate(
        **inputs,
        max_length=max_length,
        num_beams=num_beams,
        early_stopping=True,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
    )
    return best_tokenizer_helsinki.decode(outputs[0], skip_special_tokens=True)

# ============================================
# TEST INFERENCE
# ============================================
test_samples_helsinki = [
    "我爱自然语言处理",
    "今天天气很好",
    "机器翻译是自然语言处理的重要任务",
    "专家们警告朝鲜继续推进武器计划",
    "我在学习深度学习",
]

print("\n" + "="*60)
print("🔤 TEST INFERENCE - HELSINKI OPUS-MT ZH-VI")
print("="*60)

for zh in test_samples_helsinki:
    vi = translate_helsinki(zh)
    print(f"ZH: {zh}")
    print(f"VI: {vi}")
    print("-" * 50)


⏳ Loading best Helsinki model for inference...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Loaded

🔤 TEST INFERENCE - HELSINKI OPUS-MT ZH-VI
ZH: 我爱自然语言处理
VI: Tôi thích xử lý ngôn ngữ tự nhiên
--------------------------------------------------
ZH: 今天天气很好
VI: Hôm nay thời tiết rất tốt
--------------------------------------------------
ZH: 机器翻译是自然语言处理的重要任务
VI: Dịch máy là nhiệm vụ quan trọng để xử lý ngôn ngữ tự nhiên
--------------------------------------------------
ZH: 专家们警告朝鲜继续推进武器计划
VI: Các chuyên gia cảnh báo Triều Tiên tiếp tục tiến hành kế hoạch vũ khí
--------------------------------------------------
ZH: 我在学习深度学习
VI: Tôi đang học hỏi sâu.
--------------------------------------------------


In [ ]:
# ============================================
# EVALUATE BLEU ON TEST SET
# ============================================
@torch.inference_mode()
def evaluate_bleu_helsinki(test_tsv_path, max_len=128, max_samples=None):
    best_model_helsinki.eval()
    hypotheses = []
    references = []
    
    with open(test_tsv_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    if max_samples:
        lines = lines[:max_samples]
    
    for line in tqdm(lines, desc='Evaluating Helsinki BLEU'):
        parts = line.strip().split('\t')
        if len(parts) != 2:
            continue
        zh, vi_ref = parts[0].strip(), parts[1].strip()
        if not zh or not vi_ref:
            continue
        
        inputs = best_tokenizer_helsinki(zh, return_tensors="pt", truncation=True, max_length=max_len).to(DEVICE)
        outputs = best_model_helsinki.generate(**inputs, max_length=max_len, num_beams=4, early_stopping=True)
        vi_hyp = best_tokenizer_helsinki.decode(outputs[0], skip_special_tokens=True)
        
        hypotheses.append(vi_hyp)
        references.append(vi_ref)
    
    bleu = sacrebleu.corpus_bleu(hypotheses, [references])
    bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="char")
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    
    print(f'\n=== KẾT QUẢ BLEU HELSINKI ({len(hypotheses):,} câu) ===')
    print(f'BLEU score     : {bleu.score:.2f}')
    print(f'BLEU char      : {bleu_char.score:.2f}')
    print(f'chrF           : {chrf.score:.2f}')
    print(f'Brevity Penalty: {bleu.bp:.4f}')
    print(f'Ratio (hyp/ref): {bleu.sys_len / bleu.ref_len:.4f}')
    
    return bleu.score

# ============================================
# RUN BLEU EVALUATION
# ============================================
print("\n⏳ Evaluating Helsinki BLEU on test set...")
helsinki_bleu = evaluate_bleu_helsinki(
    str("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/test.tsv")
)

print(f"\n✅ Helsinki Final BLEU score: {helsinki_bleu:.2f}")


⏳ Evaluating Helsinki BLEU on test set...


Evaluating Helsinki BLEU:   0%|          | 0/3000 [00:00<?, ?it/s]


=== KẾT QUẢ BLEU HELSINKI (3,000 câu) ===
BLEU score     : 39.93
BLEU char      : 62.66
chrF           : 57.02
Brevity Penalty: 0.9736
Ratio (hyp/ref): 0.9739

✅ Helsinki Final BLEU score: 39.93


In [ ]:
# ============================================
# LƯU KẾT QUẢ
# ============================================
results_helsinki = {
    "model_name": MODEL_NAME_HELSINKI,
    "train_samples": len(df_train_helsinki),
    "dev_samples": len(df_dev_helsinki),
    "test_samples": len(df_test_helsinki),
    "bleu_score": helsinki_bleu,
    "num_epochs": training_args_helsinki.num_train_epochs,
    "batch_size": training_args_helsinki.per_device_train_batch_size,
    "learning_rate": training_args_helsinki.learning_rate,
    "effective_batch": training_args_helsinki.per_device_train_batch_size * training_args_helsinki.gradient_accumulation_steps,
    "checkpoints_dir": str(HELSINKI_CHECKPOINT_DIR),
}

with open(HELSINKI_OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results_helsinki, f, ensure_ascii=False, indent=2)

print(f"\n✅ Results saved to: {HELSINKI_OUTPUT_DIR / 'results.json'}")

print("\n" + "="*60)
print("✅ HELSINKI OPUS-MT ZH-VI FINISHED!")
print(f"   Model saved to: {HELSINKI_BEST_DIR}")
print(f"   Checkpoints saved to: {HELSINKI_CHECKPOINT_DIR}")
print("="*60)

# In tóm tắt
print("\n📊 TÓM TẮT KẾT QUẢ:")
print(f"   BLEU score: {helsinki_bleu:.2f}")
print(f"   Train samples: {len(df_train_helsinki):,}")
print(f"   Epochs: {training_args_helsinki.num_train_epochs}")
print(f"   Batch size: {training_args_helsinki.per_device_train_batch_size}")
print(f"   Learning rate: {training_args_helsinki.learning_rate}")


✅ Results saved to: /kaggle/working/zh_vi_translation/helsinki_output/results.json

✅ HELSINKI OPUS-MT ZH-VI FINISHED!
   Model saved to: /kaggle/working/zh_vi_translation/helsinki_output/best_model
   Checkpoints saved to: /kaggle/working/zh_vi_translation/helsinki_output/checkpoints

📊 TÓM TẮT KẾT QUẢ:
   BLEU score: 43.87
   Train samples: 494,000
   Epochs: 5
   Batch size: 16
   Learning rate: 5e-05


#### V. CHUẨN BỊ TẬP ĐÁNH GIÁ VLSP 2022

##### 5.1. Cấu hình đường dẫn VLSP 2022

In [64]:
from pathlib import Path
import pandas as pd

VLSP_ZH = Path("/kaggle/input/datasets/esonip/vlsp2022/vlsp2022_test/test.vi-zh.2022.zh")
VLSP_VI = Path("/kaggle/input/datasets/esonip/vlsp2022/vlsp2022_test/test.vi-zh.2022.vi")

VLSP_TSV = Path("/kaggle/working/zh_vi_translation/vlsp2022_test/vlsp2022_zh_vi_test.tsv")
VLSP_TSV.parent.mkdir(parents=True, exist_ok=True)

print("VLSP_ZH:", VLSP_ZH, "| exists:", VLSP_ZH.exists())
print("VLSP_VI:", VLSP_VI, "| exists:", VLSP_VI.exists())
print("VLSP_TSV:", VLSP_TSV)

assert VLSP_ZH.exists(), f"Không thấy file tiếng Trung: {VLSP_ZH}"
assert VLSP_VI.exists(), f"Không thấy file tiếng Việt: {VLSP_VI}"

VLSP_ZH: /kaggle/input/datasets/esonip/vlsp2022/vlsp2022_test/test.vi-zh.2022.zh | exists: True
VLSP_VI: /kaggle/input/datasets/esonip/vlsp2022/vlsp2022_test/test.vi-zh.2022.vi | exists: True
VLSP_TSV: /kaggle/working/zh_vi_translation/vlsp2022_test/vlsp2022_zh_vi_test.tsv


##### 5.2. Đọc file nguồn tiếng Trung và tham chiếu tiếng Việt

In [61]:
def read_lines(path):
    with open(path, "r", encoding="utf-8") as f:
        return [line.strip() for line in f]

vlsp_zh_lines = read_lines(VLSP_ZH)
vlsp_vi_lines = read_lines(VLSP_VI)

print("Số dòng zh:", len(vlsp_zh_lines))
print("Số dòng vi:", len(vlsp_vi_lines))

assert len(vlsp_zh_lines) == len(vlsp_vi_lines), (
    f"Lệch số dòng: zh={len(vlsp_zh_lines)}, vi={len(vlsp_vi_lines)}"
)

Số dòng zh: 1000
Số dòng vi: 1000


##### 5.3. Kiểm tra số lượng và ví dụ dữ liệu

In [62]:
vlsp_pairs = [
    (zh, vi)
    for zh, vi in zip(vlsp_zh_lines, vlsp_vi_lines)
    if zh and vi
]

print(f"Tổng cặp câu VLSP 2022 sau khi bỏ dòng rỗng: {len(vlsp_pairs):,}")

for i, (zh, vi) in enumerate(vlsp_pairs[:5], start=1):
    print(f"\n--- Sample {i} ---")
    print("ZH :", zh)
    print("VI :", vi)

Tổng cặp câu VLSP 2022 sau khi bỏ dòng rỗng: 1,000

--- Sample 1 ---
ZH : 在遭受新冠肺炎疫情不小影响的背景下，2022年对越南恢复和发展经济具有重要意义。
VI : Năm 2022 là năm có ý nghĩa quan trọng trong phục hồi và phát triển kinh tế Việt Nam sau những tác động của dịch COVID - 19.

--- Sample 2 ---
ZH : 利用好现有优势将助力越南恢复发展势头。
VI : Tận dụng tốt những lợi thế sẵn có sẽ giúp Việt Nam nhanh chóng quay lại đà tăng trưởng.

--- Sample 3 ---
ZH : 国会会议上，国会代表们集中分析，提出多项促进经济社会发展的措施，助力越南实现可持续发展。
VI : Trên diễn đàn Quốc hội, các đại biểu đã phân tích, đưa ra nhiều giải pháp phát triển kinh tế - xã hội năm 2022 để Việt Nam phát triển bền vững.

--- Sample 4 ---
ZH : 与新冠肺炎病毒共处不仅是医疗卫生问题，而且还涉及国家经济及社会所有活动的运作方式。
VI : Sống chung với dịch COVID-19 không chỉ là vấn đề về y tế mà còn liên quan tới cả cách thức vận hành nền kinh tế và mọi hoạt động của xã hội.

--- Sample 5 ---
ZH : 国会代表就预测后疫情时期情况及出台与之适应的战略和措施提供意见建议。
VI : Các đại biểu Quốc hội đã cùng góp ý để định hình, dự báo về tương lai sau đại dịch Covid-19 và có chiến lược, giải pháp phù hợp.


##### 5.4. Chuyển VLSP 2022 sang định dạng TSV

In [65]:
vlsp_df = pd.DataFrame(vlsp_pairs, columns=["zh", "vi"])

vlsp_df.to_csv(
    VLSP_TSV,
    sep="\t",
    index=False,
    header=False,
    encoding="utf-8",
)

print("Đã lưu VLSP TSV:", VLSP_TSV)
print("Số dòng:", len(vlsp_df))

display(vlsp_df.head())

Đã lưu VLSP TSV: /kaggle/working/zh_vi_translation/vlsp2022_test/vlsp2022_zh_vi_test.tsv
Số dòng: 1000


,zh,vi
0,在遭受新冠肺炎疫情不小影响的背景下，2022年对越南恢复和发展经济具有重要意义。,Năm 2022 là năm có ý nghĩa quan trọng trong ph...
1,利用好现有优势将助力越南恢复发展势头。,Tận dụng tốt những lợi thế sẵn có sẽ giúp Việt...
2,国会会议上，国会代表们集中分析，提出多项促进经济社会发展的措施，助力越南实现可持续发展。,"Trên diễn đàn Quốc hội, các đại biểu đã phân t..."
3,与新冠肺炎病毒共处不仅是医疗卫生问题，而且还涉及国家经济及社会所有活动的运作方式。,Sống chung với dịch COVID-19 không chỉ là vấn ...
4,国会代表就预测后疫情时期情况及出台与之适应的战略和措施提供意见建议。,Các đại biểu Quốc hội đã cùng góp ý để định hì...


#### VI. EVALUTE TRÊN TẬP VLSP 2022

In [2]:
!pip -q install sacrebleu sentencepiece opencc-python-reimplemented

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 26.5 MB/s eta 0:00:00


##### 6.1. Config chung

In [4]:
from pathlib import Path
import json, gc
import pandas as pd
import torch
import sacrebleu
import sentencepiece as spm
from tqdm.auto import tqdm
import random
import re
import unicodedata

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Đường dẫn trên Kaggle
DATA_DIR = Path("/kaggle/working/zh_vi_translation")

# VLSP 2022 dataset
VLSP_TSV = Path("/kaggle/input/datasets/esonip/vlsp2022/vlsp2022_test/vlsp2022_zh_vi_test.tsv")

# Transformer paths
TF_BEST_DIR = Path("/kaggle/input/models/esonip/transformer-model/transformers/v1/1/transformer_output/transformer_best.pt")
TF_SPM_ZH = Path("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_zh.model")
TF_SPM_VI = Path("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_vi.model")

# Hirashiba paths
HIRASHIBA_BEST_DIR = Path("/kaggle/input/models/esonip/hirashiba-model/transformers/v1/1/hirashiba_output/best_model")

# Helsinki paths
HELSINKI_BEST_DIR = Path("/kaggle/input/models/esonip/helsinki-model/transformers/v1/1/helsinki_output/best_model")

# Output evaluation
EVAL_OUTPUT_DIR = DATA_DIR / "vlsp2022_eval"
EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra file tồn tại
print("\n📁 KIỂM TRA ĐƯỜNG DẪN:")
for name, path in {
    "VLSP_TSV": VLSP_TSV,
    "TF_BEST_DIR": TF_BEST_DIR,
    "TF_SPM_ZH": TF_SPM_ZH,
    "TF_SPM_VI": TF_SPM_VI,
    "HIRASHIBA_BEST_DIR": HIRASHIBA_BEST_DIR,
    "HELSINKI_BEST_DIR": HELSINKI_BEST_DIR,
}.items():
    print(f"   {name}: {path} | exists: {path.exists()}")

Device: cuda

📁 KIỂM TRA ĐƯỜNG DẪN:
   VLSP_TSV: /kaggle/input/datasets/esonip/vlsp2022/vlsp2022_test/vlsp2022_zh_vi_test.tsv | exists: True
   TF_BEST_DIR: /kaggle/input/models/esonip/transformer-model/transformers/v1/1/transformer_output/transformer_best.pt | exists: True
   TF_SPM_ZH: /kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_zh.model | exists: True
   TF_SPM_VI: /kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_vi.model | exists: True
   HIRASHIBA_BEST_DIR: /kaggle/input/models/esonip/hirashiba-model/transformers/v1/1/hirashiba_output/best_model | exists: True
   HELSINKI_BEST_DIR: /kaggle/input/models/esonip/helsinki-model/transformers/v1/1/helsinki_output/best_model | exists: True


##### 6.2. Load VLSP + hàm tính BLEU

In [5]:
assert VLSP_TSV.exists(), f"Không thấy VLSP_TSV: {VLSP_TSV}"

vlsp_df = pd.read_csv(
    VLSP_TSV,
    sep="\t",
    header=None,
    names=["zh", "vi"],
    quoting=3,
    dtype=str,
    engine="python",
).dropna().reset_index(drop=True)

vlsp_df["zh"] = vlsp_df["zh"].astype(str).str.strip()
vlsp_df["vi"] = vlsp_df["vi"].astype(str).str.strip()
vlsp_df = vlsp_df[(vlsp_df["zh"] != "") & (vlsp_df["vi"] != "")].reset_index(drop=True)

vlsp_sources = vlsp_df["zh"].tolist()
vlsp_refs = vlsp_df["vi"].tolist()
vlsp_pairs = list(zip(vlsp_sources, vlsp_refs))

print(f"VLSP2022: {len(vlsp_df):,} câu")
display(vlsp_df.head())


def clear_cuda():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def calc_bleu_objects(hypotheses, references):
    bleu_13a = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="13a")
    bleu_char = sacrebleu.corpus_bleu(hypotheses, [references], tokenize="char")
    chrf = sacrebleu.corpus_chrf(hypotheses, [references])
    return bleu_13a, bleu_char, chrf


def print_bleu_report(model_name, hypotheses, references, sources, sample_k=5):
    import random

    bleu_13a, bleu_char, chrf = calc_bleu_objects(hypotheses, references)

    print("\n" + "=" * 60)
    print(f"  KẾT QUẢ BLEU - {model_name}")
    print(f"  TẬP TEST VLSP 2022 ({len(hypotheses):,} câu)")
    print("=" * 60)
    print(f"  BLEU score     : {bleu_13a.score:.2f}")
    print(f"  Brevity Penalty: {bleu_13a.bp:.4f}")
    print(f"  Ratio (hyp/ref): {bleu_13a.sys_len / bleu_13a.ref_len:.4f}")
    print(
        f"  n-gram precisions: "
        f"1-gram={bleu_13a.precisions[0]:.1f}  "
        f"2-gram={bleu_13a.precisions[1]:.1f}  "
        f"3-gram={bleu_13a.precisions[2]:.1f}  "
        f"4-gram={bleu_13a.precisions[3]:.1f}"
    )
    print(f"  BLEU char      : {bleu_char.score:.2f}")
    print(f"  chrF           : {chrf.score:.2f}")
    print("=" * 60)

    print(f"\n--- Ví dụ ngẫu nhiên ({min(sample_k, len(hypotheses))} câu) ---")
    if hypotheses:
        indices = random.sample(range(len(hypotheses)), min(sample_k, len(hypotheses)))
        for i in indices:
            print(f"ZH : {sources[i]}")
            print(f"REF: {references[i]}")
            print(f"HYP: {hypotheses[i]}")
            print("-" * 50)

    return bleu_13a, bleu_char, chrf


eval_rows = []
all_predictions = {}


def save_result(model_key, model_name, hypotheses, references, sources, pretrained, checkpoint, sample_k=5):
    bleu_13a, bleu_char, chrf = print_bleu_report(
        model_name=model_name,
        hypotheses=hypotheses,
        references=references,
        sources=sources,
        sample_k=sample_k,
    )

    row = {
        "model_key": model_key,
        "model_name": model_name,
        "pretrained": pretrained,
        "checkpoint": str(checkpoint),
        "test_set": "VLSP2022 zh-vi",
        "num_sentences": len(hypotheses),
        "bleu_13a": bleu_13a.score,
        "bleu_char": bleu_char.score,
        "chrf": chrf.score,
        "bp": bleu_13a.bp,
        "ratio": bleu_13a.sys_len / bleu_13a.ref_len,
        "precisions": list(bleu_13a.precisions),
    }

    pred_df = pd.DataFrame({
        "zh": sources,
        "reference_vi": references,
        "hypothesis_vi": hypotheses,
    })

    pred_path = EVAL_OUTPUT_DIR / f"{model_key}_vlsp2022_predictions.tsv"
    result_path = EVAL_OUTPUT_DIR / f"{model_key}_vlsp2022_metrics.json"

    pred_df.to_csv(pred_path, sep="\t", index=False, encoding="utf-8")

    with open(result_path, "w", encoding="utf-8") as f:
        json.dump({
            **row,
            "sample_hyp": hypotheses[:20],
            "sample_ref": references[:20],
        }, f, ensure_ascii=False, indent=2)

    eval_rows.append(row)
    all_predictions[model_key] = pred_df

    print(f"\n✅ Đã lưu metrics: {result_path}")
    print(f"✅ Đã lưu predictions: {pred_path}")

VLSP2022: 1,000 câu


,zh,vi
0,在遭受新冠肺炎疫情不小影响的背景下，2022年对越南恢复和发展经济具有重要意义。,Năm 2022 là năm có ý nghĩa quan trọng trong ph...
1,利用好现有优势将助力越南恢复发展势头。,Tận dụng tốt những lợi thế sẵn có sẽ giúp Việt...
2,国会会议上，国会代表们集中分析，提出多项促进经济社会发展的措施，助力越南实现可持续发展。,"Trên diễn đàn Quốc hội, các đại biểu đã phân t..."
3,与新冠肺炎病毒共处不仅是医疗卫生问题，而且还涉及国家经济及社会所有活动的运作方式。,Sống chung với dịch COVID-19 không chỉ là vấn ...
4,国会代表就预测后疫情时期情况及出台与之适应的战略和措施提供意见建议。,Các đại biểu Quốc hội đã cùng góp ý để định hì...


##### 6.3. Evaluate Transformer

In [ ]:
# ============================================
# EVALUATE TRANSFORMER
# ============================================

print("\n" + "="*60)
print("🚀 EVALUATING TRANSFORMER...")
print("="*60)

TRANSFORMER_CKPT_PATH = Path("/kaggle/input/models/esonip/transformer-model/transformers/v1/1/transformer_output/transformer_best.pt")
TRANSFORMER_SPM_ZH = Path("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_zh.model")
TRANSFORMER_SPM_VI = Path("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_vi.model")

if TRANSFORMER_CKPT_PATH.exists() and TRANSFORMER_SPM_ZH.exists():
    print(f"⏳ Loading Transformer...")
    
    # Load tokenizer
    sp_zh = spm.SentencePieceProcessor()
    sp_vi = spm.SentencePieceProcessor()
    sp_zh.load(str(TRANSFORMER_SPM_ZH))
    sp_vi.load(str(TRANSFORMER_SPM_VI))
    
    # Load model
    model = Seq2SeqTransformer(
        src_vocab_size=sp_zh.get_piece_size(),
        tgt_vocab_size=sp_vi.get_piece_size(),
    ).to(DEVICE)
    
    ckpt = torch.load(TRANSFORMER_CKPT_PATH, map_location=DEVICE)
    state_dict = ckpt['model_state']
    if any(k.startswith("_orig_mod.") for k in state_dict.keys()):
        state_dict = {k.replace("_orig_mod.", "", 1): v for k, v in state_dict.items()}
    model.load_state_dict(state_dict)
    model.eval()
    print(f"✅ Loaded: epoch={ckpt.get('epoch')}, dev_loss={ckpt.get('dev_loss'):.4f}")
    
    # Hàm dịch
    @torch.inference_mode()
    def translate_transformer(text):
        src_ids = sp_zh.encode(text) + [EOS_ID]
        src = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
        src_emb = model.pos_enc(model.src_emb(src) * model.scale)
        src_pad_mask = model.make_src_key_padding_mask(src)
        memory = model.transformer.encoder(src_emb, src_key_padding_mask=src_pad_mask)
        tgt_ids = [BOS_ID]
        for _ in range(MAX_LEN):
            tgt = torch.tensor([tgt_ids], dtype=torch.long, device=DEVICE)
            tgt_emb = model.pos_enc(model.tgt_emb(tgt) * model.scale)
            tgt_mask = model.make_tgt_mask(tgt)
            out = model.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask, memory_key_padding_mask=src_pad_mask)
            next_id = model.fc_out(out[:, -1, :]).argmax(-1).item()
            if next_id == EOS_ID:
                break
            tgt_ids.append(next_id)
        return sp_vi.decode(tgt_ids[1:]).strip()
    
    # Dịch
    hypotheses = []
    for text in tqdm(vlsp_sources, desc="Transformer translating"):
        try:
            hypotheses.append(translate_transformer(text))
        except:
            hypotheses.append("")
    
    # Lưu kết quả
    save_result(
        model_key="transformer",
        model_name="Transformer",
        hypotheses=hypotheses,
        references=vlsp_refs,
        sources=vlsp_sources,
        checkpoint=TRANSFORMER_CKPT_PATH,
        pretrained="No",
    )
    clear_cuda()
else:
    print(f"⚠️ Transformer model not found!")


🚀 EVALUATING TRANSFORMER...
⏳ Loading Transformer...
✅ Loaded: epoch=26, dev_loss=2.6212


Transformer translating:   0%|          | 0/1000 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at /pytorch/aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(



  KẾT QUẢ BLEU - Transformer
  TẬP TEST VLSP 2022 (1,000 câu)
  BLEU score     : 24.45
  Brevity Penalty: 0.9248
  Ratio (hyp/ref): 0.9274
  n-gram precisions: 1-gram=59.2  2-gram=33.7  3-gram=19.8  4-gram=12.4
  BLEU char      : 50.28
  chrF           : 45.28

--- Ví dụ ngẫu nhiên (5 câu) ---
ZH : 只有每个人都健康，每家企业都健康，国家才会健康。
REF: Chỉ khi mọi người khỏe mạnh và mọi công ty khỏe mạnh thì đất nước mới khỏe mạnh được.
HYP: Chỉ có mỗi người đều khỏe mạnh, mỗi doanh nghiệp đều khỏe mạnh, quốc gia mới có thể khỏe mạnh.
--------------------------------------------------
ZH : 在塔利班接管喀布尔后，纳伊姆强调，塔利班实现了其20多年来所追求的“国家自由和人民独立”目标。
REF: Sau khi Taliban đánh chiếm Kabul, Naim nhấn mạnh rằng Taliban đã đạt được mục tiêu “tự do dân tộc và độc lập của nhân dân” mà họ theo đuổi hơn 20 năm.
HYP: "Sau khi Taliban chiếm Kabul, ông Nayim nhấn mạnh Taliban đã đạt được mục tiêu ""tự do quốc gia và độc lập của người dân"" mà họ đã theo đuổi trong hơn 20 năm qua."
--------------------------------------------------
ZH

##### 6.4. Evaluate Hirashiba

In [ ]:
# ============================================
# EVALUATE HIRASHIBA
# ============================================

print("\n" + "="*60)
print("🚀 EVALUATING HIRASHIBA...")
print("="*60)

if HIRASHIBA_BEST_DIR.exists():
    print(f"⏳ Loading Hirashiba from {HIRASHIBA_BEST_DIR}...")
    
    hirashiba_tokenizer = AutoTokenizer.from_pretrained(str(HIRASHIBA_BEST_DIR), trust_remote_code=True)
    hirashiba_model = AutoModelForSeq2SeqLM.from_pretrained(str(HIRASHIBA_BEST_DIR), trust_remote_code=True).to(DEVICE)
    hirashiba_model.eval()
    print("✅ Hirashiba loaded")
    
    @torch.inference_mode()
    def translate_hirashiba(text):
        inputs = hirashiba_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        outputs = hirashiba_model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
        return hirashiba_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Translate all
    hypotheses = []
    for text in tqdm(vlsp_sources, desc="Hirashiba translating"):
        try:
            hypotheses.append(translate_hirashiba(text))
        except:
            hypotheses.append("")
    
    save_result(
        model_key="hirashiba",
        model_name="Hirashiba-MT-tiny-zh-vi (fine-tuned)",
        hypotheses=hypotheses,
        references=vlsp_refs,
        sources=vlsp_sources,
        checkpoint=HIRASHIBA_BEST_DIR,
        pretrained="Yes",
    )
    clear_cuda()
else:
    print(f"⚠️ Hirashiba model not found at {HIRASHIBA_BEST_DIR}")


🚀 EVALUATING HIRASHIBA...
⏳ Loading Hirashiba from /kaggle/input/models/esonip/hirashiba-model/transformers/v1/1/hirashiba_output/best_model...


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

✅ Hirashiba loaded


Hirashiba translating:   0%|          | 0/1000 [00:00<?, ?it/s]


  KẾT QUẢ BLEU - Hirashiba-MT-tiny-zh-vi (fine-tuned)
  TẬP TEST VLSP 2022 (1,000 câu)
  BLEU score     : 27.81
  Brevity Penalty: 0.9258
  Ratio (hyp/ref): 0.9284
  n-gram precisions: 1-gram=62.7  2-gram=37.4  3-gram=23.1  4-gram=15.0
  BLEU char      : 53.78
  chrF           : 48.57

--- Ví dụ ngẫu nhiên (5 câu) ---
ZH : 落实政府总理范明政关于加强胡志明市及南部各省新冠肺炎疫情防控力量的指示，自2021年8月20日起，来自全国各地的大批干部、军队指战员、公安力量、医务人员纷纷支援南方各省防疫。
REF: Thực hiện Chỉ thị của Thủ tướng Chính phủ Phạm Minh Chính về tăng cường các lực lượng tham gia phòng, chống dịch Covid-19 trên địa bàn Thành phố Hồ Chí Minh và các tỉnh phía Nam, từ ngày 20/8/2021, nhiều lượt cán bộ, chiến sỹ quân đội, công an, đội ngũ y, bác sỹ ở mọi miền Tổ quốc đã gấp rút chi viện cho các tỉnh phía Nam.
HYP: Thực hiện chỉ thị của Thủ tướng Chính phủ về việc tăng cường quyền kiểm soát dịch bệnh viêm phổi mới tại các tỉnh của Thành phố Hồ Chí Minh và miền Nam, kể từ ngày 20 tháng 8 năm 2021, nhiều cán bộ, quân đội, lực lượng công an, nhân viên y tế từ khắp 

##### 6.5. Evaluate Helsinki OPUS-MT

In [ ]:
# ============================================
# EVALUATE HELSINKI
# ============================================

print("\n" + "="*60)
print("🚀 EVALUATING HELSINKI...")
print("="*60)

if HELSINKI_BEST_DIR.exists():
    print(f"⏳ Loading Helsinki from {HELSINKI_BEST_DIR}...")
    
    # Load tokenizer và model (dùng Marian)
    helsinki_tokenizer = MarianTokenizer.from_pretrained(str(HELSINKI_BEST_DIR))
    helsinki_model = MarianMTModel.from_pretrained(str(HELSINKI_BEST_DIR)).to(DEVICE)
    helsinki_model.eval()
    print("✅ Helsinki loaded")
    
    # Hàm dịch 
    @torch.inference_mode()
    def translate_helsinki(text):
        inputs = helsinki_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
        outputs = helsinki_model.generate(
            **inputs, 
            max_length=128, 
            num_beams=4, 
            early_stopping=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
        )
        return helsinki_tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Translate all
    hypotheses = []
    for text in tqdm(vlsp_sources, desc="Helsinki translating"):
        try:
            hypotheses.append(translate_helsinki(text))
        except Exception as e:
            print(f"Error: {e}")
            hypotheses.append("")
    
    # Lưu kết quả 
    save_result(
        model_key="helsinki",
        model_name="Helsinki-NLP/opus-mt-zh-vi (fine-tuned)",
        hypotheses=hypotheses,
        references=vlsp_refs,
        sources=vlsp_sources,
        checkpoint=HELSINKI_BEST_DIR,
        pretrained="Yes",
    )
    clear_cuda()
else:
    print(f"⚠️ Helsinki model not found at {HELSINKI_BEST_DIR}")


🚀 EVALUATING HELSINKI...
⏳ Loading Helsinki from /kaggle/input/models/esonip/helsinki-model/transformers/v1/1/helsinki_output/best_model...


/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Helsinki loaded


Helsinki translating:   0%|          | 0/1000 [00:00<?, ?it/s]


  KẾT QUẢ BLEU - Helsinki-NLP/opus-mt-zh-vi (fine-tuned)
  TẬP TEST VLSP 2022 (1,000 câu)
  BLEU score     : 23.26
  Brevity Penalty: 0.9043
  Ratio (hyp/ref): 0.9086
  n-gram precisions: 1-gram=59.6  2-gram=33.2  3-gram=19.0  4-gram=11.6
  BLEU char      : 49.17
  chrF           : 44.36

--- Ví dụ ngẫu nhiên (5 câu) ---
ZH : 他老人家最特别的地方是朴素和真诚。
REF: Nét đặc biệt ở Người là sự khiêm tốn, giản dị, chân thành.
HYP: Điểm đặc biệt nhất của ông cụ là sự phát triển và chân thành.
--------------------------------------------------
ZH : 保护和维持企业安全生产经营活动是经济复苏涉及的第一要素。
REF: Chủ trương bảo vệ, hỗ trợ duy trì hoạt động sản xuất kinh doanh an toàn của doanh nghiệp trong điều kiện sống chung với dịch là yếu tố nhắc đến trước tiên trong khôi phục kinh tế.
HYP: Bảo vệ và duy trì hoạt động sản xuất an toàn doanh nghiệp là yếu tố đầu tiên liên quan đến phục hồi kinh tế.
--------------------------------------------------
ZH : 对此，国会文化、教育和青少年儿童委员会前副主任阮曰职认为，党员不自我锻炼，反而道德、生活作风堕落，那么将滑向“自我演变”、“自我转化”。
REF: Về vấn đ

In [18]:
import json
import pandas as pd
from pathlib import Path

EVAL_OUTPUT_DIR = Path("/kaggle/input/datasets/esonip/vlsp2022-evaluate/vlsp2022_eval")

metrics_files = {
    "Custom Transformer": EVAL_OUTPUT_DIR / "transformer_vlsp2022_metrics.json",
    "Hirashiba-MT-tiny": EVAL_OUTPUT_DIR / "hirashiba_vlsp2022_metrics.json",
    "Helsinki OPUS-MT": EVAL_OUTPUT_DIR / "helsinki_vlsp2022_metrics.json",
}

# Đọc kết quả từ file
results = {}
for name, path in metrics_files.items():
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
            results[name] = data
        print(f"✅ Đã đọc: {name}")
    else:
        print(f"⚠️ Không tìm thấy: {path}")
        results[name] = None

print("\n" + "="*80)
print("📊 BẢNG SO SÁNH KẾT QUẢ 3 MODEL TRÊN VLSP 2022")
print("="*80)

# Tạo bảng so sánh
comparison_data = []

for name, data in results.items():
    if data:
        comparison_data.append({
            "Model": name,
            "BLEU (13a)": f"{data.get('bleu_score', data.get('bleu_13a', 0)):.2f}",
            "BLEU (char)": f"{data.get('bleu_char', 0):.2f}",
            "chrF": f"{data.get('chrf', 0):.2f}",
            "Brevity Penalty": f"{data.get('brevity_penalty', data.get('bp', 0)):.4f}",
            "Ratio (hyp/ref)": f"{data.get('ratio', 0):.4f}",
            "Test samples": data.get('test_samples', data.get('num_sentences', 0)),
        })
    else:
        comparison_data.append({
            "Model": name,
            "BLEU (13a)": "N/A",
            "BLEU (char)": "N/A",
            "chrF": "N/A",
            "Brevity Penalty": "N/A",
            "Ratio (hyp/ref)": "N/A",
            "Test samples": "N/A",
        })

# Tạo DataFrame
df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))

✅ Đã đọc: Custom Transformer
✅ Đã đọc: Hirashiba-MT-tiny
✅ Đã đọc: Helsinki OPUS-MT

📊 BẢNG SO SÁNH KẾT QUẢ 3 MODEL TRÊN VLSP 2022
             Model BLEU (13a) BLEU (char)  chrF Brevity Penalty Ratio (hyp/ref)  Test samples
Custom Transformer      24.45       50.28 45.28          0.9248          0.9274          1000
 Hirashiba-MT-tiny      27.81       53.78 48.57          0.9258          0.9284          1000
  Helsinki OPUS-MT      23.26       49.17 44.36          0.9043          0.9086          1000


In [20]:
# ============================================
# IN CHI TIẾT TỪNG MODEL
# ============================================
print("\n" + "="*80)
print("📈 CHI TIẾT TỪNG MODEL")
print("="*80)

for name, data in results.items():
    if data:
        print(f"\n🔹 {name}")
        print(f"   ├─ BLEU (13a)     : {data.get('bleu_score', data.get('bleu_13a', 0)):.2f}")
        print(f"   ├─ BLEU (char)    : {data.get('bleu_char', 0):.2f}")
        print(f"   ├─ chrF           : {data.get('chrf', 0):.2f}")
        print(f"   ├─ Brevity Penalty: {data.get('brevity_penalty', data.get('bp', 0)):.4f}")
        print(f"   ├─ Ratio (hyp/ref): {data.get('ratio', 0):.4f}")
        print(f"   └─ Test samples   : {data.get('test_samples', data.get('num_sentences', 0))}")
    else:
        print(f"\n🔹 {name}: ⚠️ Chưa có kết quả")

# ============================================
# TÌM MODEL TỐT NHẤT
# ============================================
print("\n" + "="*80)
print("🏆 XẾP HẠNG THEO BLEU SCORE")
print("="*80)

# Lọc các model có dữ liệu
valid_results = [(name, data) for name, data in results.items() if data is not None]

if valid_results:
    # Sắp xếp theo BLEU score giảm dần
    sorted_results = sorted(
        valid_results, 
        key=lambda x: x[1].get('bleu_score', x[1].get('bleu_13a', 0)), 
        reverse=True
    )
    
    for i, (name, data) in enumerate(sorted_results, 1):
        bleu = data.get('bleu_score', data.get('bleu_13a', 0))
        chrf = data.get('chrf', 0)
        medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(i, f"{i}.")
        print(f"   {medal} {name}: BLEU={bleu:.2f}, chrF={chrf:.2f}")
else:
    print("   ⚠️ Chưa có kết quả để xếp hạng")

# ============================================
# LƯU BẢNG SO SÁNH
# ============================================

comparison_path = Path("/kaggle/working/zh_vi_translation/vlsp2022_eval/model_comparison.csv")
df_comparison.to_csv(comparison_path, index=False)
print(f"\n✅ Đã lưu bảng so sánh tại: {comparison_path}")

print("\n" + "="*80)


📈 CHI TIẾT TỪNG MODEL

🔹 Custom Transformer
   ├─ BLEU (13a)     : 24.45
   ├─ BLEU (char)    : 50.28
   ├─ chrF           : 45.28
   ├─ Brevity Penalty: 0.9248
   ├─ Ratio (hyp/ref): 0.9274
   └─ Test samples   : 1000

🔹 Hirashiba-MT-tiny
   ├─ BLEU (13a)     : 27.81
   ├─ BLEU (char)    : 53.78
   ├─ chrF           : 48.57
   ├─ Brevity Penalty: 0.9258
   ├─ Ratio (hyp/ref): 0.9284
   └─ Test samples   : 1000

🔹 Helsinki OPUS-MT
   ├─ BLEU (13a)     : 23.26
   ├─ BLEU (char)    : 49.17
   ├─ chrF           : 44.36
   ├─ Brevity Penalty: 0.9043
   ├─ Ratio (hyp/ref): 0.9086
   └─ Test samples   : 1000

🏆 XẾP HẠNG THEO BLEU SCORE
   🥇 Hirashiba-MT-tiny: BLEU=27.81, chrF=48.57
   🥈 Custom Transformer: BLEU=24.45, chrF=45.28
   🥉 Helsinki OPUS-MT: BLEU=23.26, chrF=44.36

✅ Đã lưu bảng so sánh tại: /kaggle/working/zh_vi_translation/vlsp2022_eval/model_comparison.csv



In [ ]:
# ============================================
# TEST CÂU BẤT KỲ
# ============================================

import torch, math
import sentencepiece as spm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, MarianTokenizer, MarianMTModel
from pathlib import Path

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================
# LOAD CUSTOM TRANSFORMER
# ============================================
# Định nghĩa lại class
class PositionalEncoding(torch.nn.Module):
    def __init__(self, d_model, max_len=128, dropout=0.1):
        super().__init__()
        self.dropout = torch.nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class Seq2SeqTransformer(torch.nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size, d_model=256, nhead=8,
                 num_encoder_layers=4, num_decoder_layers=4, dim_feedforward=1024,
                 dropout=0.1, max_len=128):
        super().__init__()
        self.src_emb = torch.nn.Embedding(src_vocab_size, d_model, padding_idx=0)
        self.tgt_emb = torch.nn.Embedding(tgt_vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        self.transformer = torch.nn.Transformer(
            d_model=d_model, nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True,
        )
        self.fc_out = torch.nn.Linear(d_model, tgt_vocab_size)
        self.scale = math.sqrt(d_model)
    def make_src_key_padding_mask(self, src):
        return src == 0
    def make_tgt_mask(self, tgt):
        return torch.nn.Transformer.generate_square_subsequent_mask(tgt.size(1), device=tgt.device)
    def forward(self, src, tgt):
        src_emb = self.pos_enc(self.src_emb(src) * self.scale)
        tgt_emb = self.pos_enc(self.tgt_emb(tgt) * self.scale)
        tgt_mask = self.make_tgt_mask(tgt)
        src_pad_mask = self.make_src_key_padding_mask(src)
        tgt_pad_mask = self.make_src_key_padding_mask(tgt)
        out = self.transformer(
            src_emb, tgt_emb,
            tgt_mask=tgt_mask,
            src_key_padding_mask=src_pad_mask,
            tgt_key_padding_mask=tgt_pad_mask,
            memory_key_padding_mask=src_pad_mask,
        )
        return self.fc_out(out)

# Load Transformer
sp_zh = spm.SentencePieceProcessor()
sp_vi = spm.SentencePieceProcessor()
sp_zh.load("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_zh.model")
sp_vi.load("/kaggle/input/datasets/esonip/zh-vi-processed/zh_vi_processed/spm_vi.model")

transformer_model = Seq2SeqTransformer(
    src_vocab_size=sp_zh.get_piece_size(),
    tgt_vocab_size=sp_vi.get_piece_size(),
).to(DEVICE)

ckpt = torch.load("/kaggle/input/models/esonip/transformer-model/transformers/v1/1/transformer_output/transformer_best.pt", map_location=DEVICE)
state_dict = ckpt['model_state']
if any(k.startswith("_orig_mod.") for k in state_dict.keys()):
    state_dict = {k.replace("_orig_mod.", "", 1): v for k, v in state_dict.items()}
transformer_model.load_state_dict(state_dict)
transformer_model.eval()

def translate_transformer(text):
    src_ids = sp_zh.encode(text) + [3]  # EOS=3
    src = torch.tensor([src_ids], dtype=torch.long, device=DEVICE)
    src_emb = transformer_model.pos_enc(transformer_model.src_emb(src) * transformer_model.scale)
    src_pad_mask = transformer_model.make_src_key_padding_mask(src)
    memory = transformer_model.transformer.encoder(src_emb, src_key_padding_mask=src_pad_mask)
    tgt_ids = [2]  # BOS=2
    for _ in range(128):
        tgt = torch.tensor([tgt_ids], dtype=torch.long, device=DEVICE)
        tgt_emb = transformer_model.pos_enc(transformer_model.tgt_emb(tgt) * transformer_model.scale)
        tgt_mask = transformer_model.make_tgt_mask(tgt)
        out = transformer_model.transformer.decoder(tgt_emb, memory, tgt_mask=tgt_mask, memory_key_padding_mask=src_pad_mask)
        next_id = transformer_model.fc_out(out[:, -1, :]).argmax(-1).item()
        if next_id == 3:
            break
        tgt_ids.append(next_id)
    return sp_vi.decode(tgt_ids[1:]).strip()

# ============================================
# LOAD HIRASHIBA
# ============================================
hirashiba_tokenizer = AutoTokenizer.from_pretrained("chi-vi/hirashiba-mt-tiny-zh-vi", trust_remote_code=True)
hirashiba_model = AutoModelForSeq2SeqLM.from_pretrained("chi-vi/hirashiba-mt-tiny-zh-vi", trust_remote_code=True).to(DEVICE)
hirashiba_model.eval()

def translate_hirashiba(text):
    inputs = hirashiba_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    outputs = hirashiba_model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
    return hirashiba_tokenizer.decode(outputs[0], skip_special_tokens=True)

# ============================================
# LOAD HELSINKI
# ============================================
helsinki_tokenizer = MarianTokenizer.from_pretrained("/kaggle/input/models/esonip/helsinki-model/transformers/v1/1/helsinki_output/best_model")
helsinki_model = MarianMTModel.from_pretrained("/kaggle/input/models/esonip/helsinki-model/transformers/v1/1/helsinki_output/best_model").to(DEVICE)
helsinki_model.eval()

def translate_helsinki(text):
    inputs = helsinki_tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(DEVICE)
    outputs = helsinki_model.generate(**inputs, max_length=128, num_beams=4, early_stopping=True)
    return helsinki_tokenizer.decode(outputs[0], skip_special_tokens=True)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/256 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [ ]:
# ============================================
# TEST
# ============================================
test_sentence = "生活是一连串的课程，必须经历才能被理解"

# Phiên âm: Shēnghuó shì yì lián chuàn de kèchéng, bìxū jīnglì cái néng bèi lǐjiě.
# Dịch: Cuộc sống là một chuỗi bài học, bạn cần phải trải nghiệm mới hiểu hết được

print("\n" + "="*60)
print(f"📝 TEST CÂU: {test_sentence}")
print("="*60)

print(f"\n🔹 Custom Transformer: {translate_transformer(test_sentence)}")
print(f"\n🔹 Hirashiba: {translate_hirashiba(test_sentence)}")
print(f"\n🔹 Helsinki: {translate_helsinki(test_sentence)}")

print("\n" + "="*60)


📝 TEST CÂU: 生活是一连串的课程，必须经历才能被理解

🔹 Custom Transformer: Cuộc sống là một chuỗi các khóa học, phải trải nghiệm để được hiểu

🔹 Hirashiba: Cuộc sống là chuỗi tiết học, phải trải qua mới thấu hiểu được

🔹 Helsinki: Cuộc sống là một chuỗi các khóa học cần phải trải nghiệm để được hiểu

